# ClinR2G-Fusion
## ResNet-18 + DenseNet-121 Unified Encoder · Cross-Attention Fusion · Clinical Focal Loss · Full-Length Generation

**Model:** ClinR2G-Fusion  
**Changes vs. prior submission:**
1. **Unified encoder** — ResNet-18 spatial branch fused with DenseNet-121 via learned cross-attention gate  
2. **Explicit cross-attention fusion layer** between encoder output and decoder (DICTA requirement)  
3. **Clinical Focal Loss formula corrected** — alpha weighting now applied consistently with the focal term  
4. **Generation cap removed** — `max_len` raised to `MAX_SEQ` (128 tokens) so full radiology reports are produced  

**Dataset:** IU X-Ray (OpenI Indiana University)  
> Runtime → Run all (T4 GPU recommended)

---

## Step 0 — Install & Imports

In [ ]:
!pip install -q nltk opencv-python-headless bert-score rouge-score

import os, re, json, random, math, warnings
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from tqdm.notebook import tqdm
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
import torchvision.models as models
from sklearn.model_selection import train_test_split

import nltk
nltk.download('punkt',   quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'  GPU : {torch.cuda.get_device_name(0)}')


## Step 1 — Mount Drive & Set Paths

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/medical_vlm_project')
except Exception:
    ROOT = Path('/content/medical_vlm_project')

IMAGE_DIR  = ROOT / 'data' / 'raw' / 'images'
REPORT_DIR = ROOT / 'data' / 'raw' / 'reports'
OUTPUT_DIR = ROOT / 'outputs'
PROCESSED  = ROOT / 'data' / 'processed'
for d in [OUTPUT_DIR, PROCESSED]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Images : {len(list(IMAGE_DIR.glob("*.png")))} PNG')
print(f'Reports: {len(list(REPORT_DIR.glob("*.xml")))} XML')


## Step 2 — Configuration

Key changes from prior notebook:
- `MAX_SEQ = 128` (was 50) — full radiology report length
- Generation functions no longer hard-cap at 20 tokens


In [ ]:
# ── Training ──────────────────────────────────────────────────────────────────
IMAGE_SIZE    = 224
BATCH_SIZE    = 32
LR            = 1e-4
WARMUP_EPOCHS = 3
NUM_EPOCHS    = 30
EARLY_STOP    = 5
GRAD_CLIP     = 0.5
USE_AMP       = torch.cuda.is_available()
MIN_WORDS     = 4
MAX_SUBSET    = None

# ── Model ─────────────────────────────────────────────────────────────────────
EMBED_DIM  = 512      # shared projection dim for both encoder branches
N_HEADS    = 8
N_LAYERS   = 3
FF_DIM     = 1024
DROPOUT    = 0.1

# FIX 4: Generation cap removed — was 20, now 128 to allow full reports
MAX_SEQ    = 128

# ── Decoding ──────────────────────────────────────────────────────────────────
BEAM_SIZE    = 3
LEN_ALPHA    = 0.6
NO_RPT_NGRAM = 3
RPT_PENALTY  = 1.2
MIN_GEN_LEN  = 10     # raised from 4 to 10 — encourages complete sentences

# ── Data balancing ────────────────────────────────────────────────────────────
ABNORMAL_OVERSAMPLE = 3.0

print(f'MAX_SEQ       = {MAX_SEQ}  (full-length generation)')
print(f'MIN_GEN_LEN   = {MIN_GEN_LEN}')
print(f'EMBED_DIM     = {EMBED_DIM}')


## Step 3 — Parse XML & Impression-Only Extraction

In [ ]:
_ABNORMAL_KW = {
    'pneumonia','effusion','cardiomegaly','atelectasis','consolidation',
    'nodule','mass','opacity','edema','pneumothorax','fracture',
    'abnormal','infiltrate','pleural','haziness','enlarged'
}
_NORMAL_KW = {
    'normal','clear','unremarkable','no acute','no evidence','without','negative'
}

def classify_report(text: str) -> int:
    t = text.lower()
    if any(k in t for k in _ABNORMAL_KW):
        return 1
    if any(k in t for k in _NORMAL_KW):
        return 0
    return -1   # unclear

def parse_xml_report(xml_path: Path) -> dict | None:
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        study_id = xml_path.stem

        # Collect all AbstractText sections
        sections = {}
        for ab in root.iter('AbstractText'):
            label = ab.get('Label', '').lower()
            text  = (ab.text or '').strip()
            if text:
                sections[label] = text

        # Prefer 'impression', fall back to 'findings'
        report_text = sections.get('impression') or sections.get('findings', '')
        if not report_text or len(report_text.split()) < MIN_WORDS:
            return None

        # Collect image filenames from parentImage
        images = []
        for pi in root.iter('parentImage'):
            uid = pi.get('id', '')
            if uid:
                images.append(f'{uid}.png')

        label = classify_report(report_text)
        return {
            'study_id'   : study_id,
            'report_text': report_text.lower().strip(),
            'images'     : images,
            'label'      : label,
        }
    except Exception:
        return None

records = []
for xml_path in sorted(REPORT_DIR.glob('*.xml')):
    r = parse_xml_report(xml_path)
    if r:
        records.append(r)

print(f'Parsed {len(records)} valid reports')

# Explode into one row per image
rows = []
for r in records:
    for img in r['images']:
        rows.append({
            'study_id'   : r['study_id'],
            'image_path' : IMAGE_DIR / img,
            'report_text': r['report_text'],
            'label'      : r['label'],
        })

df = pd.DataFrame(rows)
df['image_exists'] = df['image_path'].apply(lambda p: p.exists())
df = df[df['image_exists']].reset_index(drop=True)

print(f'Total image–report pairs: {len(df)}')
print(f"Label distribution:\n{df['label'].value_counts()}")


## Step 4 — Study-Level Split (Leakage Prevention)

In [ ]:
unique_ids = df['study_id'].unique()
train_ids, temp_ids = train_test_split(unique_ids, test_size=0.2, random_state=SEED)
val_ids,   test_ids = train_test_split(temp_ids,   test_size=0.5, random_state=SEED)

train_df = df[df['study_id'].isin(train_ids)].reset_index(drop=True)
val_df   = df[df['study_id'].isin(val_ids)].reset_index(drop=True)
test_df  = df[df['study_id'].isin(test_ids)].reset_index(drop=True)

# Sanity: no overlap
assert not set(train_ids) & set(val_ids), 'LEAK: train/val overlap'
assert not set(train_ids) & set(test_ids), 'LEAK: train/test overlap'
assert not set(val_ids)   & set(test_ids), 'LEAK: val/test overlap'

print(f'Train: {len(train_df)} rows ({train_df["study_id"].nunique()} studies)')
print(f'Val  : {len(val_df)} rows ({val_df["study_id"].nunique()} studies)')
print(f'Test : {len(test_df)} rows ({test_df["study_id"].nunique()} studies)')


## Step 5 — Vocabulary Construction

In [ ]:
class Vocab:
    PAD, SOS, EOS, UNK = 0, 1, 2, 3

    def __init__(self, min_freq=2):
        self.min_freq  = min_freq
        self.word2idx  = {'<pad>':0,'<sos>':1,'<eos>':2,'<unk>':3}
        self.idx2word  = {v:k for k,v in self.word2idx.items()}
        self._counter  = Counter()
        self._finalised = False

    def update(self, text: str):
        assert not self._finalised
        self._counter.update(re.findall(r"[a-z0-9.,]+", text.lower()))

    def finalise(self):
        for word, freq in self._counter.items():
            if freq >= self.min_freq and word not in self.word2idx:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx]  = word
        self._finalised = True
        print(f'Vocabulary size: {len(self.word2idx)}')

    def encode(self, text, max_len=MAX_SEQ):
        tokens = re.findall(r"[a-z0-9.,]+", text.lower())
        ids    = [self.SOS] + [self.word2idx.get(t, self.UNK) for t in tokens[:max_len-2]] + [self.EOS]
        ids   += [self.PAD] * (max_len - len(ids))
        return ids[:max_len]

    def decode(self, ids):
        words = []
        for i in ids:
            if i in (self.PAD, self.SOS): continue
            if i == self.EOS: break
            words.append(self.idx2word.get(i, '<unk>'))
        return ' '.join(words)

    def __len__(self): return len(self.word2idx)

vocab = Vocab(min_freq=2)
for text in train_df['report_text']:
    vocab.update(text)
vocab.finalise()


## Step 6 — Dataset, Transforms & Weighted Sampler

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class CXRDataset(Dataset):
    def __init__(self, dataframe, vocab, transform):
        self.df        = dataframe.reset_index(drop=True)
        self.vocab     = vocab
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        img   = Image.open(row['image_path']).convert('RGB')
        img   = self.transform(img)
        toks  = torch.tensor(self.vocab.encode(row['report_text']), dtype=torch.long)
        label = int(row['label']) if row['label'] != -1 else 0
        return img, toks, label, str(row['image_path'])


def collate_fn(batch):
    imgs, toks, labels, paths = zip(*batch)
    return (torch.stack(imgs),
            torch.stack(toks),
            torch.tensor(labels),
            list(paths))


# Weighted sampler for class imbalance
label_counts = train_df['label'].value_counts()
weights = train_df['label'].map(
    lambda l: ABNORMAL_OVERSAMPLE if l == 1 else 1.0
).values
sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

train_ds = CXRDataset(train_df, vocab, train_transform)
val_ds   = CXRDataset(val_df,   vocab, val_transform)
test_ds  = CXRDataset(test_df,  vocab, val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          collate_fn=collate_fn, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=2, pin_memory=True)

# Quick sanity
imgs, toks, labels, _ = next(iter(train_loader))
print(f'Image batch : {imgs.shape}')
print(f'Token batch : {toks.shape}')
print(f'Label batch : {labels.shape}')


## Step 7 — ClinR2G-Fusion Architecture

### FIX 1 — Unified ResNet-18 + DenseNet-121 Encoder

Both backbones extract spatial feature maps at 7×7. A **cross-attention gating**
layer fuses them before handing off to the decoder. ResNet-18 provides fine edge
and texture details; DenseNet-121 (the CheXNet backbone) contributes dense
multi-scale features tuned to chest pathology.

### FIX 2 — Explicit Cross-Attention Fusion Layer (DICTA requirement)

`CrossAttentionFusion` uses DenseNet features as *queries* and ResNet features
as *keys/values* (and vice-versa in the second head), then linearly combines
both attended outputs. This is separate from the Transformer decoder's own
cross-attention, giving an intermediate encoder-level fusion signal.


In [ ]:
# FIX 1 & 2: Unified encoder = ResNet-18 + DenseNet-121 + CrossAttentionFusion

class ResNetBranch(nn.Module):
    """ResNet-18 spatial branch. Strips the global pool + FC head.
    Returns [B, 49, EMBED_DIM] feature tokens."""
    def __init__(self, embed_dim=EMBED_DIM):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        # Keep everything up to (but not including) avgpool
        self.backbone = nn.Sequential(*list(resnet.children())[:-2])  # → (B,512,7,7)

        # Freeze early layers (layer1 & layer2), keep layer3 & layer4 trainable
        freeze_layers = ['conv1','bn1','relu','maxpool','layer1','layer2']
        for name, module in resnet.named_children():
            if name in freeze_layers:
                for p in module.parameters():
                    p.requires_grad = False

        self.proj = nn.Sequential(
            nn.Linear(512, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU(),
            nn.Dropout(DROPOUT),
        )
        self.pool = nn.AdaptiveAvgPool2d((7, 7))

    def forward(self, x):
        feat = self.backbone(x)           # (B, 512, H, W)
        feat = self.pool(feat)            # (B, 512, 7, 7)
        B, C, H, W = feat.shape
        feat = feat.permute(0,2,3,1).reshape(B, H*W, C)   # (B, 49, 512)
        return self.proj(feat)            # (B, 49, EMBED_DIM)


class DenseNetBranch(nn.Module):
    """DenseNet-121 (CheXNet backbone) spatial branch.
    Returns [B, 49, EMBED_DIM] feature tokens."""
    def __init__(self, embed_dim=EMBED_DIM):
        super().__init__()
        densenet = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        self.features = densenet.features  # → (B, 1024, H, W)

        # Freeze early blocks (conv0 → denseblock2)
        freeze_prefixes = ['conv0','norm0','relu0','pool0',
                           'denseblock1','transition1','denseblock2','transition2']
        for name, param in self.features.named_parameters():
            if any(name.startswith(p) for p in freeze_prefixes):
                param.requires_grad = False

        self.pool = nn.AdaptiveAvgPool2d((7, 7))
        self.proj = nn.Sequential(
            nn.Linear(1024, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU(),
            nn.Dropout(DROPOUT),
        )

    def forward(self, x):
        feat = self.features(x)           # (B, 1024, H, W)
        feat = F.relu(feat)               # DenseNet final ReLU
        feat = self.pool(feat)            # (B, 1024, 7, 7)
        B, C, H, W = feat.shape
        feat = feat.permute(0,2,3,1).reshape(B, H*W, C)   # (B, 49, 1024)
        return self.proj(feat)            # (B, 49, EMBED_DIM)


class CrossAttentionFusion(nn.Module):
    """
    FIX 2 — Explicit cross-attention fusion between encoder branches.

    Two cross-attention passes:
      Pass A: DenseNet as Query, ResNet as Key/Value  (what DenseNet attends to in ResNet)
      Pass B: ResNet   as Query, DenseNet as Key/Value (what ResNet attends to in DenseNet)

    Fused output = learnable convex combination of the two attended outputs
    plus a residual skip from mean of both raw branches.

    Shape: (B, 49, EMBED_DIM) → (B, 49, EMBED_DIM)
    """
    def __init__(self, embed_dim=EMBED_DIM, n_heads=N_HEADS):
        super().__init__()
        self.attn_d2r = nn.MultiheadAttention(embed_dim, n_heads,
                                               dropout=DROPOUT, batch_first=True)
        self.attn_r2d = nn.MultiheadAttention(embed_dim, n_heads,
                                               dropout=DROPOUT, batch_first=True)
        self.gate = nn.Parameter(torch.tensor(0.5))   # learnable mix weight
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ffn   = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 2),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(embed_dim * 2, embed_dim),
        )
        self.norm3 = nn.LayerNorm(embed_dim)

    def forward(self, dense_feat, resnet_feat):
        # Pass A: DenseNet queries into ResNet context
        a_out, _ = self.attn_d2r(query=dense_feat,
                                   key=resnet_feat,
                                   value=resnet_feat)
        # Pass B: ResNet queries into DenseNet context
        b_out, _ = self.attn_r2d(query=resnet_feat,
                                   key=dense_feat,
                                   value=dense_feat)

        g = torch.sigmoid(self.gate)           # scalar in (0,1)
        fused = g * self.norm1(a_out) + (1.0 - g) * self.norm2(b_out)

        # Residual skip from raw average
        fused = fused + 0.5 * (dense_feat + resnet_feat)

        # Position-wise FFN
        fused = fused + self.ffn(self.norm3(fused))
        return fused


class UnifiedEncoder(nn.Module):
    """
    FIX 1 + FIX 2 combined:
    Runs ResNet-18 and DenseNet-121 in parallel, then fuses with CrossAttentionFusion.
    Returns (B, 49, EMBED_DIM) visual tokens.
    """
    def __init__(self, embed_dim=EMBED_DIM):
        super().__init__()
        self.resnet_branch  = ResNetBranch(embed_dim)
        self.densenet_branch = DenseNetBranch(embed_dim)
        self.fusion          = CrossAttentionFusion(embed_dim, N_HEADS)

    def forward(self, x):
        r_feat = self.resnet_branch(x)    # (B, 49, EMBED_DIM)
        d_feat = self.densenet_branch(x)  # (B, 49, EMBED_DIM)
        return self.fusion(d_feat, r_feat) # (B, 49, EMBED_DIM)


# Transformer Decoder (uses PyTorch's built-in cross-attention between decoder tokens and encoder memory)
# ─────────────────────────────────────────────────────────────────────────────

class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=EMBED_DIM):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_emb = nn.Embedding(MAX_SEQ, embed_dim)
        self.drop    = nn.Dropout(DROPOUT)

        layer = nn.TransformerDecoderLayer(
            d_model=embed_dim, nhead=N_HEADS, dim_feedforward=FF_DIM,
            dropout=DROPOUT, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerDecoder(layer, num_layers=N_LAYERS)
        self.norm    = nn.LayerNorm(embed_dim)
        self.fc_out  = nn.Linear(embed_dim, vocab_size)

    def forward(self, tokens, memory):
        B, T = tokens.shape
        pos  = torch.arange(T, device=tokens.device).unsqueeze(0)
        x    = self.drop(self.tok_emb(tokens) + self.pos_emb(pos))
        mask = nn.Transformer.generate_square_subsequent_mask(T, device=tokens.device)
        pad  = tokens.eq(0)
        out  = self.transformer(x, memory, tgt_mask=mask, tgt_key_padding_mask=pad)
        return self.fc_out(self.norm(out))


# Full model

class ClinR2GFusion(nn.Module):
    """
    ClinR2G-Fusion: unified dual-backbone encoder + cross-attention fusion
    + Transformer decoder.

    Forward path:
      image → ResNet-18 branch → (B,49,D)
                                       ↘
                                         CrossAttentionFusion → memory (B,49,D)
                                       ↗                              ↓
      image → DenseNet-121 branch → (B,49,D)          Transformer decoder
                                                              ↓
                                                     logits (B, T, V)
    """
    def __init__(self, vocab_size):
        super().__init__()
        self.encoder = UnifiedEncoder(EMBED_DIM)
        self.decoder = TransformerDecoder(vocab_size, EMBED_DIM)

    def forward(self, images, tokens):
        memory = self.encoder(images)        # (B, 49, EMBED_DIM)
        return self.decoder(tokens, memory)  # (B, T, V)


model = ClinR2GFusion(vocab_size=len(vocab)).to(device)
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'ClinR2G-Fusion model ready')
print(f'  Total params    : {total:,}')
print(f'  Trainable params: {trainable:,}')
print(f'  Frozen params   : {total-trainable:,}')

# Forward pass check
model.eval()
with torch.no_grad():
    out = model(imgs.to(device), toks[:, :-1].to(device))
print(f'  Forward pass OK : {out.shape}  (B, T, V)')


## Step 8 — Clinical Focal Loss (Fixed)

### FIX 3 — Formula corrected

1. `alpha` values are now normalised so they live in `(0, 1]`.
2. The formula strictly follows Lin et al. 2017:  
   `FL(p_t) = -alpha_t · (1 - p_t)^gamma · log(p_t)`  
   where `alpha_t` scales *class frequency correction* and `(1-p_t)^gamma`
   scales *example difficulty*.  Clinical terms get a higher `alpha_t` (i.e.
   they are treated as a rarer, more important class), which is correct.
3. Special tokens (`<pad>`, `<sos>`, `<eos>`, `<unk>`) keep `alpha_t = 0`
   so they are completely excluded from the gradient.


In [ ]:
class ClinicalFocalLoss(nn.Module):

    def __init__(self, vocab, gamma: float = 2.0,
                 ignore_index: int = 0, clinical_boost: float = 3.0):
        super().__init__()
        self.gamma        = gamma
        self.ignore_index = ignore_index

        CLINICAL_TERMS = {
            'pneumonia','effusion','cardiomegaly','atelectasis',
            'consolidation','nodule','mass','opacity','edema',
            'pneumothorax','fracture','abnormal','infiltrate',
            'pleural','haziness','enlarged','atelectatic'
        }
        SPECIAL_TOKENS = {'<pad>','<sos>','<eos>','<unk>'}

        # Build raw alpha vector
        raw_alpha = torch.ones(len(vocab))
        for word, idx in vocab.word2idx.items():
            if word in SPECIAL_TOKENS:
                raw_alpha[idx] = 0.0          # excluded from loss
            elif word in CLINICAL_TERMS:
                raw_alpha[idx] = clinical_boost  # rare-class boost

        # Normalise so all non-zero weights lie in (0, 1]
        # Divide by the maximum non-zero value → clinical terms get alpha = 1.0,
        # regular words get alpha = 1/clinical_boost ≈ 0.33 (with boost=3).
        max_val = raw_alpha[raw_alpha > 0].max()
        alpha   = raw_alpha / max_val          # in [0, 1]

        self.register_buffer('alpha', alpha)   # (V,)

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """
        Args:
            logits  : (N, V)  raw unnormalised scores
            targets : (N,)    ground-truth token indices
        Returns:
            scalar focal loss (mean over valid tokens)
        """
        # Mask out pad / special tokens using ignore_index (PAD = 0)
        mask = targets.ne(self.ignore_index)
        if mask.sum() == 0:
            return logits.sum() * 0.0         # safe zero gradient

        logits  = logits[mask]                # (M, V)
        targets = targets[mask]               # (M,)

        # ── Core focal loss ──────────────────────────────────────────────────
        log_probs = F.log_softmax(logits, dim=-1)            # (M, V)
        ce_loss   = F.nll_loss(log_probs, targets,           # (M,)
                                reduction='none')
        p_t       = torch.exp(-ce_loss)                      # (M,)  ∈ (0,1]
        focal_wt  = (1.0 - p_t) ** self.gamma               # (M,)
        alpha_t   = self.alpha[targets]                      # (M,)

        # FL = alpha_t · (1-p_t)^gamma · ce_loss
        loss = (alpha_t * focal_wt * ce_loss).mean()
        return loss


criterion = ClinicalFocalLoss(
    vocab          = vocab,
    gamma          = 2.0,
    ignore_index   = vocab.PAD,
    clinical_boost = 3.0,
)
criterion.to(device)
print('Clinical Focal Loss (fixed) ready')
print(f'  gamma          = 2.0')
print(f'  clinical_boost = 3.0  (normalised to alpha=1.0 for clinical terms)')
print(f'  regular words  = alpha ≈ {1.0/3.0:.3f}')
print(f'  special tokens = alpha = 0.0  (excluded)')

# ── Quick sanity test 
with torch.no_grad():
    dummy_logits  = torch.randn(32, len(vocab)).to(device)
    dummy_targets = torch.randint(1, len(vocab), (32,)).to(device)
    dummy_targets[0] = vocab.PAD
    test_loss = criterion(dummy_logits, dummy_targets)
print(f'  Sanity test loss: {test_loss.item():.4f}  ✓')


## Step 9 — Training with Warmup + Cosine Schedule

In [ ]:
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=1e-4)

def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(NUM_EPOCHS - WARMUP_EPOCHS, 1)
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler    = torch.cuda.amp.GradScaler(enabled=USE_AMP)


def run_epoch(loader, training=True):
    model.train() if training else model.eval()
    total_loss, total_tok = 0.0, 0
    pbar = tqdm(loader, leave=False)
    for images, tokens, _, _ in pbar:
        images = images.to(device, non_blocking=True)
        tokens = tokens.to(device, non_blocking=True)
        inputs, targets = tokens[:, :-1], tokens[:, 1:]

        with torch.set_grad_enabled(training):
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                logits = model(images, inputs)
                B, S, V = logits.shape
                loss = criterion(logits.reshape(B*S, V), targets.reshape(B*S))

            if training:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()

        n = targets.ne(vocab.PAD).sum().item()
        total_loss += loss.item() * max(n, 1)
        total_tok  += max(n, 1)
        pbar.set_description(f'{"tr" if training else "vl"} {loss.item():.3f}')

    return total_loss / max(total_tok, 1)


BEST_PATH = OUTPUT_DIR / 'best_clinr2g_fusion.pth'
best_val  = float('inf')
patience  = 0
history   = {'train': [], 'val': []}

for epoch in range(NUM_EPOCHS):
    t_loss = run_epoch(train_loader, training=True)
    with torch.no_grad():
        v_loss = run_epoch(val_loader, training=False)
    scheduler.step()

    history['train'].append(t_loss)
    history['val'].append(v_loss)

    lr_now = optimizer.param_groups[0]['lr']
    print(f'Epoch {epoch+1:02d}/{NUM_EPOCHS} | '
          f'train {t_loss:.4f} | val {v_loss:.4f} | lr {lr_now:.2e}')

    if v_loss < best_val:
        best_val = v_loss
        patience = 0
        torch.save(model.state_dict(), BEST_PATH)
        print(f'  ✓ Best model saved  (val={best_val:.4f})')
    else:
        patience += 1
        if patience >= EARLY_STOP:
            print(f'  Early stop at epoch {epoch+1}')
            break

print(f'\nBest val loss: {best_val:.4f}')


## Step 10 — Training Curves

In [ ]:
epochs = range(1, len(history['train']) + 1)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(epochs, history['train'], label='Train Loss')
ax.plot(epochs, history['val'],   label='Val Loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('ClinR2G-Fusion — Training Curves')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


## Step 11 — Report Generation

### FIX 4 — 20-token cap removed

The original `safe_greedy_short` was hard-capped at `max_len=20`, which is
far too short for real radiology reports (median ≈ 40–60 tokens on IU X-Ray).
Both greedy and beam-search decoders now default to `MAX_SEQ` (128 tokens),
with early stopping on `<eos>` and n-gram repetition penalty — no hard cap.


In [ ]:
# FIX 4: Generation with NO hard token cap — uses MAX_SEQ (128) as upper bound

@torch.no_grad()
def greedy_generate(model, img_t, vocab, max_len=MAX_SEQ):
    """
    Greedy decoding with bigram-blocking repetition penalty.
    max_len defaults to MAX_SEQ=128 (was hard-capped at 20 previously).
    """
    model.eval()
    img    = img_t.unsqueeze(0).to(device)
    memory = model.encoder(img)                           # (1,49,D)
    tokens = torch.tensor([[vocab.SOS]], dtype=torch.long, device=device)
    gen    = []
    seen_bigrams = set()

    for step in range(max_len):
        logits = model.decoder(tokens, memory)[0, -1, :]  # (V,)

        # Suppress pad
        logits[vocab.PAD] = -1e9

        # Suppress EOS for first MIN_GEN_LEN steps
        if step < MIN_GEN_LEN:
            logits[vocab.EOS] = -1e9

        # Bigram blocking: penalise repeated (prev_tok, candidate) pairs
        if gen:
            for tok_id in range(len(vocab)):
                if (gen[-1], tok_id) in seen_bigrams:
                    logits[tok_id] = -1e9

        nxt = logits.argmax(-1).item()
        if nxt == vocab.EOS:
            break

        if gen:
            seen_bigrams.add((gen[-1], nxt))
        gen.append(nxt)
        tokens = torch.cat([tokens,
                            torch.tensor([[nxt]], dtype=torch.long, device=device)],
                           dim=1)

    return vocab.decode(gen)


@torch.no_grad()
def beam_generate(model, img_t, vocab,
                  beam_size=BEAM_SIZE, max_len=MAX_SEQ,
                  len_alpha=LEN_ALPHA, no_rpt_ngram=NO_RPT_NGRAM):
    """
    Beam search with length normalisation and n-gram repetition penalty.
    max_len defaults to MAX_SEQ=128 (was hard-capped at 20 previously).
    """
    model.eval()
    img    = img_t.unsqueeze(0).to(device)
    memory = model.encoder(img)    # (1,49,D)

    # Each beam: (token_list, log_prob, done_flag)
    beams = [([vocab.SOS], 0.0, False)]

    for step in range(max_len):
        if all(b[2] for b in beams):
            break
        candidates = []
        for seq, score, done in beams:
            if done:
                candidates.append((seq, score, True))
                continue

            toks   = torch.tensor([seq], dtype=torch.long, device=device)
            logits = model.decoder(toks, memory)[0, -1, :]

            logits[vocab.PAD] = -1e9
            if step < MIN_GEN_LEN:
                logits[vocab.EOS] = -1e9

            # N-gram repetition penalty
            if no_rpt_ngram > 0 and len(seq) >= no_rpt_ngram:
                forbidden = set()
                for i in range(len(seq) - no_rpt_ngram + 1):
                    ngram = tuple(seq[i:i + no_rpt_ngram - 1])
                    forbidden.add(ngram + (seq[-1],))   # context n-gram
                for forbidden_ngram in forbidden:
                    last_n = tuple(seq[-(no_rpt_ngram-1):])
                    if last_n == forbidden_ngram[:no_rpt_ngram-1]:
                        logits[forbidden_ngram[-1]] -= math.log(RPT_PENALTY)

            log_probs = F.log_softmax(logits, dim=-1)
            top_vals, top_ids = log_probs.topk(beam_size)

            for lp, tok in zip(top_vals.tolist(), top_ids.tolist()):
                new_seq   = seq + [tok]
                new_score = score + lp
                is_done   = (tok == vocab.EOS)
                candidates.append((new_seq, new_score, is_done))

        # Length-normalised beam selection
        def norm_score(item):
            seq, score, done = item
            length = max(len(seq) - 1, 1)
            return score / (length ** len_alpha)

        beams = sorted(candidates, key=norm_score, reverse=True)[:beam_size]

    best_seq = beams[0][0]
    return vocab.decode([t for t in best_seq if t not in (vocab.SOS,)])


print(f'Greedy decoder ready (max_len={MAX_SEQ})')
print(f'Beam   decoder ready (max_len={MAX_SEQ}, beam={BEAM_SIZE})')


## Step 12 — Quantitative Evaluation

In [ ]:
from nltk.translate.bleu_score   import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score as meteor_fn
from rouge_score import rouge_scorer as rouge_lib
import numpy as np

# Load best checkpoint
model.load_state_dict(torch.load(BEST_PATH, map_location=device))
model.eval()
print(f'Loaded best checkpoint: {BEST_PATH.name}')

def lcs_len(a, b):
    m, n = len(a), len(b)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(1, m+1):
        for j in range(1, n+1):
            if a[i-1] == b[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    return dp[m][n]

def compute_rougeL(hyp_toks, ref_toks):
    l = lcs_len(hyp_toks, ref_toks)
    if l == 0: return 0.0
    p = l / len(hyp_toks) if hyp_toks else 0
    r = l / len(ref_toks)  if ref_toks  else 0
    return 2*p*r/(p+r) if (p+r) > 0 else 0.0

def evaluate(dataset, num_samples=200, method='greedy'):
    smoothie   = SmoothingFunction().method4
    rouge_sc   = rouge_lib.RougeScorer(['rouge1','rougeL'], use_stemmer=True)
    refs_corpus, hyps_corpus = [], []
    rouge1_all, rougeL_all, meteor_all = [], [], []

    indices = np.random.choice(len(dataset), min(num_samples, len(dataset)), replace=False)

    for idx in indices:
        img_t, tok_t, _, _ = dataset[idx]
        if method == 'beam':
            hyp = beam_generate(model, img_t, vocab)
        else:
            hyp = greedy_generate(model, img_t, vocab)

        ref = vocab.decode(tok_t.tolist())
        hyp_toks = hyp.split()
        ref_toks = ref.split()

        refs_corpus.append([ref_toks])
        hyps_corpus.append(hyp_toks)

        sc = rouge_sc.score(ref, hyp)
        rouge1_all.append(sc['rouge1'].fmeasure)
        rougeL_all.append(sc['rougeL'].fmeasure)
        meteor_all.append(meteor_fn([ref_toks], hyp_toks))

    bleu4 = corpus_bleu(refs_corpus, hyps_corpus,
                        weights=(0.25,)*4,
                        smoothing_function=smoothie)

    return {
        'BLEU-4' : round(bleu4, 4),
        'ROUGE-1': round(float(np.mean(rouge1_all)), 4),
        'ROUGE-L': round(float(np.mean(rougeL_all)), 4),
        'METEOR' : round(float(np.mean(meteor_all)), 4),
    }

print('Evaluating (greedy, 200 samples) ...')
results_greedy = evaluate(test_ds, num_samples=200, method='greedy')
print('Greedy:', results_greedy)

print('Evaluating (beam, 200 samples) ...')
results_beam = evaluate(test_ds, num_samples=200, method='beam')
print('Beam  :', results_beam)


## Step 13 — Benchmark Comparison

In [ ]:
import pandas as pd

comparison = pd.DataFrame([
    {'Model': 'ClinR2G-Fusion (Ours — Greedy)', 'Encoder': 'ResNet-18 + DenseNet-121',
     'Decoder': 'Transformer+CrossAttn',
     **results_greedy},
    {'Model': 'ClinR2G-Fusion (Ours — Beam)',   'Encoder': 'ResNet-18 + DenseNet-121',
     'Decoder': 'Transformer+CrossAttn',
     **results_beam},
    {'Model': 'ClinR2G-Stage2 (prior)',          'Encoder': 'DenseNet-121',
     'Decoder': 'Transformer',
     'BLEU-4': 0.076, 'ROUGE-1': 0.289, 'ROUGE-L': 0.276, 'METEOR': 0.207},
    {'Model': 'R2Gen (Chen et al. 2020)',        'Encoder': 'ResNet-101',
     'Decoder': 'Relational Memory',
     'BLEU-4': 0.167, 'ROUGE-1': None, 'ROUGE-L': 0.362, 'METEOR': 0.183},
    {'Model': 'KERM (2026)',                     'Encoder': 'ViT-B/16',
     'Decoder': 'Transformer',
     'BLEU-4': 0.182, 'ROUGE-1': None, 'ROUGE-L': 0.388, 'METEOR': 0.197},
])

display(comparison.set_index('Model'))


## Step 14 — Qualitative Examples

In [ ]:
import matplotlib.pyplot as plt

model.eval()
for i in range(3):
    img_t, tok_t, _, path = test_ds[i]

    greedy_text = greedy_generate(model, img_t, vocab)
    beam_text   = beam_generate(model, img_t, vocab)
    ref_text    = vocab.decode(tok_t.tolist())

    # De-normalise for display
    img_np = img_t.permute(1,2,0).numpy()
    img_np = img_np * [0.229,0.224,0.225] + [0.485,0.456,0.406]
    img_np = img_np.clip(0,1)

    plt.figure(figsize=(5,5))
    plt.imshow(img_np); plt.axis('off')
    plt.title(f'Test sample {i}')
    plt.show()

    print(f'Reference    : {ref_text}')
    print(f'Greedy output: {greedy_text}')
    print(f'Beam output  : {beam_text}')
    print('='*80)


## Step 15 — Encoder Interpretability (Grad-CAM on DenseNet Branch)

In [ ]:
import cv2

def grad_cam_densenet(model, img_t, device):
    """Grad-CAM on the last DenseNet-121 dense block for visualisation."""
    model.eval()
    activations, gradients = {}, {}

    def fwd_hook(m, inp, out):
        activations['feat'] = out.detach()

    def bwd_hook(m, gin, gout):
        gradients['feat'] = gout[0].detach()

    target_layer = model.encoder.densenet_branch.features.denseblock4
    fh = target_layer.register_forward_hook(fwd_hook)
    bh = target_layer.register_full_backward_hook(bwd_hook)

    img = img_t.unsqueeze(0).to(device)
    img.requires_grad_(False)

    memory = model.encoder(img)
    # Use sum of memory as a scalar proxy for what the encoder attends to
    score  = memory.sum()
    model.zero_grad()
    score.backward()

    fh.remove(); bh.remove()

    # Pool gradients over spatial dims
    act  = activations['feat']         # (1, C, H, W)
    grad = gradients['feat']           # (1, C, H, W)
    weights = grad.mean(dim=(2,3), keepdim=True)  # (1,C,1,1)
    cam  = F.relu((weights * act).sum(dim=1)).squeeze(0).cpu().numpy()

    # Resize to image size
    cam  = cv2.resize(cam, (IMAGE_SIZE, IMAGE_SIZE))
    cam  = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam


fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for col, idx in enumerate(range(3)):
    img_t, tok_t, _, _ = test_ds[idx]

    cam = grad_cam_densenet(model, img_t, device)

    img_np = img_t.permute(1,2,0).numpy()
    img_np = img_np * [0.229,0.224,0.225] + [0.485,0.456,0.406]
    img_np = img_np.clip(0,1)

    axes[0, col].imshow(img_np)
    axes[0, col].set_title(f'Sample {idx} — Original')
    axes[0, col].axis('off')

    axes[1, col].imshow(img_np)
    axes[1, col].imshow(cam, cmap='jet', alpha=0.45)
    axes[1, col].set_title(f'Sample {idx} — Grad-CAM')
    axes[1, col].axis('off')

plt.suptitle('ClinR2G-Fusion — DenseNet-Branch Grad-CAM', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


## Step 16 — Ablation Study

This ablation isolates the contribution of each change introduced in this notebook.
All variants share the same training schedule, dataset split, and evaluation protocol.

| Variant | Encoder | Focal Loss | Max tokens |
|---------|---------|-----------|------------|
| A — Stage-2 baseline | DenseNet-121 only | Original (inconsistent alpha) | 20 |
| B — FIX 1+2 only | ResNet-18+DenseNet-121+CrossAttn | Original | 20 |
| C — FIX 3 only | DenseNet-121 only | Fixed (normalised alpha) | 20 |
| D — FIX 4 only | DenseNet-121 only | Original | 128 |
| **E — All fixes (ours)** | ResNet-18+DenseNet-121+CrossAttn | Fixed | 128 |


In [ ]:
# NOTE: Run a brief training pass (5 epochs) per variant for ablation.
# Full 30-epoch training for each variant is recommended for the camera-ready.

import copy, pandas as pd

def quick_ablation_train(model_class, model_kwargs, criterion_fn,
                         n_epochs=5, label=''):
    m = model_class(**model_kwargs).to(device)
    opt = optim.AdamW(filter(lambda p: p.requires_grad, m.parameters()),
                      lr=LR, weight_decay=1e-4)
    best_v = float('inf')
    best_state = None
    for ep in range(n_epochs):
        m.train()
        for imgs_b, toks_b, _, _ in train_loader:
            imgs_b = imgs_b.to(device); toks_b = toks_b.to(device)
            inp, tgt = toks_b[:, :-1], toks_b[:, 1:]
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                lg = m(imgs_b, inp)
                B,S,V = lg.shape
                loss = criterion_fn(lg.reshape(B*S,V), tgt.reshape(B*S))
            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(m.parameters(), GRAD_CLIP)
            opt.step()
        m.eval()
        vl = 0.0; vn = 0
        with torch.no_grad():
            for imgs_b, toks_b, _, _ in val_loader:
                imgs_b = imgs_b.to(device); toks_b = toks_b.to(device)
                inp, tgt = toks_b[:,:-1], toks_b[:,1:]
                lg = m(imgs_b, inp); B,S,V = lg.shape
                vl += criterion_fn(lg.reshape(B*S,V), tgt.reshape(B*S)).item()
                vn += 1
        vl /= max(vn,1)
        if vl < best_v:
            best_v = vl
            best_state = copy.deepcopy(m.state_dict())
    m.load_state_dict(best_state)
    return m, best_v

print('Ablation training — 5 epochs each variant (extend for camera-ready)...')

# Variant E: Full model (already trained above) — reuse
results_ablation = [{'Variant': 'E — All fixes (ours)', **results_beam}]

# To add other variants: instantiate appropriate model + criterion and call
# quick_ablation_train, then evaluate() — see full ablation scaffold below.
print('Full ablation scaffold: define variants A–D as needed.')
print('Current ablation results:')
pd.DataFrame(results_ablation)


## Step 17 — Summary of Changes

| # | Issue | Root Cause | Fix Applied |
|---|-------|-----------|-------------|
| 1 | Single-backbone encoder (DenseNet-121 only) | Stage-1 used ResNet-18; Stage-2 dropped it | Added `ResNetBranch` in parallel; outputs (B,49,D) |
| 2 | No explicit cross-attention fusion layer | Reviewer / DICTA requirement | `CrossAttentionFusion` module with 2-head MHA; learnable gate parameter |
| 3 | Clinical Focal Loss alpha inconsistency | `alpha` values > 1 mixed with focal term, amplifying high-confidence predictions | Normalised `alpha` to (0,1]; formula now exactly matches Lin et al. 2017 |
| 4 | 20-token generation cap | Hard-coded `max_len=20` in `safe_greedy_short` | Both `greedy_generate` and `beam_generate` default to `MAX_SEQ=128`; early-stop on `<eos>` |
